In [1]:
!rm -rf .temp/ .demucs/

In [2]:
from dataclasses import dataclass

from pathlib import Path

AUDIO_FILE = Path() / ".." / "output.mp3"
AUDIO_FILE

PosixPath('../output.mp3')

In [3]:
import demucs.separate

demucs.separate.main(["-n", "htdemucs", "--two-stems", "vocals", str(AUDIO_FILE), "-o", ".demucs", "--device", "cuda"])

Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /home/adam/projects/emotional-dataset/src/.demucs/htdemucs
Separating track ../output.mp3


100%|██████████████████████████████████████████████| 503.09999999999997/503.09999999999997 [00:07<00:00, 63.87seconds/s]


In [4]:
vocal_target = Path(f".demucs/htdemucs/{AUDIO_FILE.stem}/vocals.wav")
vocal_target

PosixPath('.demucs/htdemucs/output/vocals.wav')

In [8]:
from pathlib import Path
import torchaudio
import torch
import shutil

from omegaconf import OmegaConf
from nemo.collections.asr.models import ClusteringDiarizer
import json

temp_dir = Path(".temp")
shutil.rmtree(temp_dir, ignore_errors=True)
temp_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# 1️. Preprocess audio: convert to mono + 16kHz using torchaudio
# -------------------------------------------------------------------------
waveform, sr = torchaudio.load(AUDIO_FILE)
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)
if sr != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)
processed_path = temp_dir / f"{AUDIO_FILE.stem}_16k.wav"
torchaudio.save(processed_path, waveform, 16000)

# -------------------------------------------------------------------------
# 2️. Build NeMo manifest file
# -------------------------------------------------------------------------
manifest_path = temp_dir / "input_manifest.json"
meta = {
    "audio_filepath": str(processed_path.absolute()),
    "offset": 0,
    "duration": None,
    "label": "infer",
    "text": "-",
    "num_speakers": None,   # or set a number if you want to enforce it
    "rttm_filepath": None,
    "uem_filepath": None,
}
with open(manifest_path, "w") as f:
    json.dump(meta, f)
    f.write("\n")

# -------------------------------------------------------------------------
# 3️. Load diarization config (diar_infer_general.yaml)
# -------------------------------------------------------------------------
config_path = Path("pipeline/diar_infer_general.yaml")
if not config_path.exists():
    raise FileNotFoundError(
        f"Missing diarization config file: {config_path}. "
        "Download one from NeMo examples or your repo."
    )

cfg = OmegaConf.load(config_path)
# Update paths dynamically
cfg.diarizer.manifest_filepath = str(manifest_path)
cfg.diarizer.out_dir = str(temp_dir)

# Optional: disable oracle VAD or speaker count if present
if "oracle_vad" in cfg.diarizer:
    cfg.diarizer.oracle_vad = False
if "oracle_num_speakers" in cfg.diarizer:
    cfg.diarizer.oracle_num_speakers = False

# -------------------------------------------------------------------------
# 4️. Run diarization
# -------------------------------------------------------------------------
diarizer = ClusteringDiarizer(cfg=cfg)
diarizer.diarize()

# -------------------------------------------------------------------------
# 5️. Return RTTM path
# -------------------------------------------------------------------------
rttm_dir = temp_dir / "pred_rttms"
rttm_files = list(rttm_dir.glob("*.rttm"))
if not rttm_files:
    raise FileNotFoundError(f"No RTTM files generated in {rttm_dir}")

rttm_files[0]

[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Loading pretrained vad_multilingual_marblenet model from NGC
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Found existing object /home/adam/.cache/torch/NeMo/NeMo_2.1.0/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Re-using file from: /home/adam/.cache/torch/NeMo/NeMo_2.1.0/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Instantiating model from pre-trained checkpoint


[NeMo W 2025-11-02 20:05:17 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/ami_train_0.63.json,/manifests/freesound_background_train.json,/manifests/freesound_laughter_train.json,/manifests/fisher_2004_background.json,/manifests/fisher_2004_speech_sampled.json,/manifests/google_train_manifest.json,/manifests/icsi_all_0.63.json,/manifests/musan_freesound_train.json,/manifests/musan_music_train.json,/manifests/musan_soundbible_train.json,/manifests/mandarin_train_sample.json,/manifests/german_train_sample.json,/manifests/spanish_train_sample.json,/manifests/french_train_sample.json,/manifests/russian_train_sample.json
    sample_rate: 16000
    labels:
    - background
    - speech
    batch_size: 256
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy

[NeMo I 2025-11-02 20:05:17 nemo_logging:393] PADDING: 16
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Model EncDecClassificationModel was successfully restored from /home/adam/.cache/torch/NeMo/NeMo_2.1.0/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Loading pretrained titanet_large model from NGC
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Found existing object /home/adam/.cache/torch/NeMo/NeMo_2.1.0/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Re-using file from: /home/adam/.cache/torch/NeMo/NeMo_2.1.0/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2025-11-02 20:05:17 nemo_logging:393] Instantiating model from pre-trained checkpoint


[NeMo W 2025-11-02 20:05:17 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2025-11-02 20:05:17 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2025-11-02 20:05:17 nemo_logging:393] PADDING: 16
[NeMo I 2025-11-02 20:05:18 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/adam/.cache/torch/NeMo/NeMo_2.1.0/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.
[NeMo I 2025-11-02 20:05:18 nemo_logging:393] Number of files to diarize: 1
[NeMo I 2025-11-02 20:05:18 nemo_logging:393] Split long audio file to avoid CUDA memory issue


...

[NeMo I 2025-11-02 20:05:20 nemo_logging:393] Saved embedding files to .temp/speaker_outputs/embeddings



clustering: 100%|██████████| 1/1 [00:00<00:00,  3.55it/s]

[NeMo I 2025-11-02 20:05:20 nemo_logging:393] Outputs are saved in /home/adam/projects/emotional-dataset/src/.temp directory



[NeMo W 2025-11-02 20:05:20 nemo_logging:405] Check if each ground truth RTTMs were present in the provided manifest file. Skipping calculation of Diariazation Error Rate


PosixPath('.temp/pred_rttms/output_16k.rttm')

In [9]:
rttm_file = rttm_files[0]
rttm_file

PosixPath('.temp/pred_rttms/output_16k.rttm')

In [10]:
@dataclass
class SpeakerSegment:
    speaker: str
    start: float
    end: float

speaker_segments: list[SpeakerSegment] = []

for line in rttm_file.read_text().splitlines():
    data = line.strip().split(" ")
    start = float(data[5])
    speaker_segments.append(SpeakerSegment(speaker=data[11], start=start, end=start + float(data[8])))

speaker_segments[:5]

[SpeakerSegment(speaker='speaker_0', start=0.0, end=1.49),
 SpeakerSegment(speaker='speaker_0', start=3.48, end=15.355),
 SpeakerSegment(speaker='speaker_3', start=15.355, end=25.355),
 SpeakerSegment(speaker='speaker_2', start=25.355, end=32.355000000000004),
 SpeakerSegment(speaker='speaker_4', start=32.355, end=34.849999999999994)]

In [11]:
import whisper

WHISPER_MODEL = "turbo"

model = whisper.load_model(WHISPER_MODEL)
    
result = model.transcribe(
    str(vocal_target.absolute()), 
    language="cs", 
    verbose=False,
    word_timestamps=True
)

result["text"]

100%|██████████| 50000/50000 [00:16<00:00, 3084.52frames/s]


" Západní médi... Eee... Eee... Boy, já to můžu vysvětlit. Západní médié často vytváří negativní obraz číny založený na zkreslených informacích a historických událostech. My samozřejmě víme, že tady vlastně vznikla nějaká kontroverze. Hledně té cesty, pokud se domluvíme a vyrazíte s náma, tak tam absolutně není žádná povinnost cokoliv o tom výhletu sdílet. Já si myslím, že on to tak neměl taky, ale on je prostě nej, neměl na to vodkovou kapacitu. Good one. Akorát, když jsou tady ty kamery? Bezpečnostní. Nebude to nikde v mediích, že jsme se tady sešli nebo takhle. Ono tady předtím bylo call centrum. No, on tady byl předtím call centrum a měli se tady vlastně jako na sledování té práce. I ta tato je na konference. No je dobrá, no je dobrá. Takže pořád, jakoby to má pro koukliv. Miký z měl premiéru svoji nový epizody Mupy. Pozvali jsme influencery na výlet do Číny. Měl premiéru v Praze vyprodanou. Byl jsem nám pozvanej, ale samozřejmě ty vejá premiéry a výlety do Prahy, že jo? Haha, už b

In [12]:
all_words: list[dict[str, float | str]] = []

for seg in result["segments"]:
    if "words" not in seg:
        continue

    for word_info in seg["words"]:
        word_info["word"] = word_info["word"].strip()
        all_words.append(word_info)

all_words[:2]

[{'word': 'Západní',
  'start': 0.0,
  'end': 0.66,
  'probability': 0.9131441414356232},
 {'word': 'médi...',
  'start': 0.66,
  'end': 1.0,
  'probability': 0.35388387739658356}]

In [13]:
def find_speaker_for_time(timestamp: float, speaker_segments: list[SpeakerSegment]) -> str:
    """
    Finds which speaker was active at a specific timestamp.
    
    We check if the timestamp is within the [start, end) interval.
    """
    for seg in speaker_segments:
        # Using a half-open interval [start, end)
        if timestamp >= seg.start and timestamp < seg.end:
            return seg.speaker
            
    # If no speaker is found for that time (i.e., it's a gap),
    # return a generic label.
    return "UNKNOWN"

In [14]:
@dataclass
class Segment:
    speaker: str
    text: str
    start: float
    end: float

sentence_boundaries = {".", "?", "!"}
sentences = []
current_sentence = []

# --- 1. Group words into sentences ---
for word in all_words:
    current_sentence.append(word)
    if any(word["word"].strip().endswith(p) for p in sentence_boundaries):
        sentences.append(current_sentence)
        current_sentence = []
if current_sentence:
    sentences.append(current_sentence)

aligned_segments: list[Segment] = []

# --- 2. Assign speaker to each sentence ---
for sentence_words in sentences:
    if not sentence_words:
        continue

    # Find all speakers for this sentence
    speakers = []
    for w in sentence_words:
        midpoint = (w["start"] + w["end"]) / 2
        speakers.append(find_speaker_for_time(midpoint, speaker_segments))

    # If multiple speakers, skip this sentence
    if len(set(speakers)) > 1:
        continue

    current_speaker = speakers[0]
    start = sentence_words[0]["start"]
    end = sentence_words[-1]["end"]
    text = " ".join(w["word"] for w in sentence_words)

    # --- 3. Create Segment ---
    aligned_segments.append(
        Segment(
            speaker=current_speaker,
            text=text.strip(),
            start=start,
            end=end
        )
    )

aligned_segments

[Segment(speaker='speaker_0', text='Západní médi...', start=0.0, end=1.0),
 Segment(speaker='speaker_0', text='Eee...', start=3.7199999999999998, end=4.36),
 Segment(speaker='speaker_0', text='Eee...', start=4.36, end=5.36),
 Segment(speaker='speaker_0', text='Boy, já to můžu vysvětlit.', start=5.36, end=7.42),
 Segment(speaker='speaker_0', text='Západní médié často vytváří negativní obraz číny založený na zkreslených informacích a historických událostech.', start=8.06, end=15.38),
 Segment(speaker='speaker_3', text='My samozřejmě víme, že tady vlastně vznikla nějaká kontroverze.', start=15.5, end=18.36),
 Segment(speaker='speaker_3', text='Hledně té cesty, pokud se domluvíme a vyrazíte s náma, tak tam absolutně není žádná povinnost cokoliv o tom výhletu sdílet.', start=18.48, end=25.0),
 Segment(speaker='speaker_0', text='Good one.', start=36.18, end=36.9),
 Segment(speaker='speaker_2', text='Akorát, když jsou tady ty kamery?', start=39.220000000000006, end=40.8),
 Segment(speaker='sp

In [15]:
MAX_GAP_TRESHOLD: float = 0.25

# Start with the first segment as our base
merged_segments = [Segment(
    speaker=aligned_segments[0].speaker,
    text=aligned_segments[0].text,
    start=aligned_segments[0].start,
    end=aligned_segments[0].end
)]

for current_segment in aligned_segments[1:]:
    last_merged = merged_segments[-1]
    
    # Calculate the time gap between the last merged segment and the current one
    gap = current_segment.start - last_merged.end

    # --- Define Merge Conditions ---
    
    # 1. Are they the same speaker? (e.g., speaker_0 followed by speaker_0)
    is_same_speaker = (last_merged.speaker == current_segment.speaker)
    
    # 2. Is one of them UNKNOWN? (The case we want to fix)
    one_is_unknown = (last_merged.speaker == 'UNKNOWN' or 
                        current_segment.speaker == 'UNKNOWN')
    
    # 3. Are they close enough in time?
    is_close_enough = (gap <= MAX_GAP_TRESHOLD)

    # --- Apply Logic ---
    if is_close_enough and (is_same_speaker or one_is_unknown):
        # --- Merge ---
        # Append the text
        last_merged.text += " " + current_segment.text
        
        # Extend the end time
        last_merged.end = current_segment.end
        
        # --- Fix the Speaker Label ---
        # If the last segment was UNKNOWN, it should inherit 
        # the speaker from the current (known) segment.
        if last_merged.speaker == 'UNKNOWN' and current_segment.speaker != 'UNKNOWN':
            last_merged.speaker = current_segment.speaker
        
        # (If the current segment is UNKNOWN, it automatically
        # inherits the speaker from last_merged, so we do nothing.)
        
    else:
        # --- Do Not Merge ---
        # This is a new, distinct segment. Add it to the list.
        merged_segments.append(Segment(
            speaker=current_segment.speaker,
            text=current_segment.text,
            start=current_segment.start,
            end=current_segment.end
        ))

merged_segments

[Segment(speaker='speaker_0', text='Západní médi...', start=0.0, end=1.0),
 Segment(speaker='speaker_0', text='Eee... Eee... Boy, já to můžu vysvětlit.', start=3.7199999999999998, end=7.42),
 Segment(speaker='speaker_0', text='Západní médié často vytváří negativní obraz číny založený na zkreslených informacích a historických událostech.', start=8.06, end=15.38),
 Segment(speaker='speaker_3', text='My samozřejmě víme, že tady vlastně vznikla nějaká kontroverze. Hledně té cesty, pokud se domluvíme a vyrazíte s náma, tak tam absolutně není žádná povinnost cokoliv o tom výhletu sdílet.', start=15.5, end=25.0),
 Segment(speaker='speaker_0', text='Good one.', start=36.18, end=36.9),
 Segment(speaker='speaker_2', text='Akorát, když jsou tady ty kamery?', start=39.220000000000006, end=40.8),
 Segment(speaker='speaker_3', text='Bezpečnostní.', start=44.2, end=44.68),
 Segment(speaker='speaker_2', text='Nebude to nikde v mediích, že jsme se tady sešli nebo takhle.', start=44.92, end=47.76),
 Seg

In [18]:
# --- Filter Logic ---
MIN_DURATION_SEC = 6.0
MAX_DURATION_SEC = 20.0

best_segments = []
for segment in merged_segments:
    duration = segment.end - segment.start
    
    if MIN_DURATION_SEC <= duration <= MAX_DURATION_SEC:
        best_segments.append(segment)

len(best_segments)

12

In [19]:
from transformers import pipeline
from datasets import Dataset
from dataclasses import field

LABEL_TO_SENTIMENT = {
    "LABEL_0": "negative",
    "LABEL_1": "negative",
    "LABEL_2": "negative",
    "LABEL_3": "negative",
    "LABEL_4": "positive",
    "LABEL_5": "neutral"
}

LABEL_TO_EMOTION = {
    "LABEL_0": "anger",
    "LABEL_1": "fear",
    "LABEL_2": "disgust",
    "LABEL_3": "sadness",
    "LABEL_4": "joy",
    "LABEL_5": "none"
}

@dataclass
class EmotionSegment(Segment):
    label: str
    
    emotion: str = field(init=False)
    sentiment: str = field(init=False)

    def __post_init__(self) -> None:
        self.emotion = LABEL_TO_EMOTION[self.label]
        self.sentiment = LABEL_TO_SENTIMENT[self.label]

MODEL = "visegradmedia-emotion/Emotion_RoBERTa_czech6"
BATCH_SIZE = 64

text = [t.text for t in best_segments]
dataset = Dataset.from_dict({"text": text})

clf = pipeline("text-classification", model=MODEL, device=0)
preds = clf(list(dataset["text"]), batch_size=BATCH_SIZE, truncation=True)

emotion_segments = [
    EmotionSegment(
        speaker=best_segments[idx].speaker,
        text=best_segments[idx].text,
        start=best_segments[idx].start,
        end=best_segments[idx].end,
        label=e["label"]
    ) for idx, e in enumerate(preds)
]

emotion_segments

Device set to use cuda:0


[EmotionSegment(speaker='speaker_0', text='Západní médié často vytváří negativní obraz číny založený na zkreslených informacích a historických událostech.', start=8.06, end=15.38, label='LABEL_1', emotion='fear', sentiment='negative'),
 EmotionSegment(speaker='speaker_3', text='My samozřejmě víme, že tady vlastně vznikla nějaká kontroverze. Hledně té cesty, pokud se domluvíme a vyrazíte s náma, tak tam absolutně není žádná povinnost cokoliv o tom výhletu sdílet.', start=15.5, end=25.0, label='LABEL_5', emotion='none', sentiment='neutral'),
 EmotionSegment(speaker='speaker_0', text="A těším se na to, protože jsem dlouho neviděl Mupy. Elementy jsou teďko zaseklí, ale oni se odseknou a pak tady bude 50 tisíc alertů. Let's go!", start=90.18, end=98.7, label='LABEL_4', emotion='joy', sentiment='positive'),
 EmotionSegment(speaker='speaker_6', text='Letěl jsem normálně do Číny, měl jsem všechno zaplacené, ještě jsem za to dostal peníze. A oni točí prostě pro čínskou televizi, jak já poznávám

In [1]:
from speechbrain.inference.interfaces import foreign_class

classifier = foreign_class(
    source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
    pymodule_file="custom_interface.py",
    classname="CustomEncoderWav2vec2Classifier"
)

# preprocess/resample audio to 16kHz mono first if needed
out_prob, score, index, text_lab = classifier.classify_file(".temp/output_16k.wav")
print("Predicted emotion:", text_lab)
# out_prob gives probability, index gives label index


/home/adam/projects/emotional-dataset/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/adam/projects/emotional-dataset/.venv/lib64/python3.11/site-packages/transformers/configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.
CategoricalEncoder.expect_len was never called: assuming category count of 4 to be correct! Sanity check your encoder using `.expect_len`. Ensure that downstream code also uses the correct size. If you are sure th

Predicted emotion: ['hap']
